# Module 41 — Object Detection Architectures

## Section 4: Practical Model Selection

<badge color="success">Must Know</badge> <badge color="info">Industry & Deployment</badge>

We've now studied several object detection architectures:

- Faster R-CNN
- SSD
- RetinaNet
- YOLO
- DETR

But here's an important engineering question:

**If you're building an object detection application, how do you decide which model to use?**

You shouldn't select a model simply because it is popular or has the highest accuracy.

A model that performs extremely well on a powerful GPU might be completely unsuitable for a low-cost industrial computer.

Model selection is an engineering trade-off between six important factors:

1. Accuracy
2. Latency
3. Model size
4. Hardware
5. Dataset size
6. Real-time requirement

Let's understand each.

---

## 1. Accuracy

Accuracy tells us how well the model detects objects.

For object detection, we commonly evaluate:

- Precision
- Recall
- AP
- mAP
- IoU-based localization quality

We already studied these in Module 40.

### Example

Suppose we're building a pedestrian detection system.

| Model | mAP@0.50:0.95 |
|---|---:|
| Model A | 0.92 |
| Model B | 0.85 |
| Model C | 0.78 |

Model A has the highest reported detection score.

Does that automatically mean we should select Model A?

**No!**

We must also consider its inference speed, memory requirements and deployment hardware.

### When should accuracy get higher priority?

- Medical image analysis.
- Detailed industrial defect inspection.
- Applications where missed detections are expensive.
- Safety-critical object detection.

![Computer vision quality inspection](https://images.openai.com/static-rsc-4/V_z15MTQnWI-UNhBTV3GyGWSYqgN7ZNWJiZKmtODhuOmlQa9IfMKrSrPv0rAJjhH2pQ5KfAiN1mEvU5ylIdemOL7Rv0ma3q8wvjt3OczRYDvp4MFbExzZ4nZMea4Y9UaTp-ErY6ZZvo0Ybu7vZvnisuQHBfqR6HM7snYMqVEab4?purpose=inline)

[Source: Computer Vision Quality Inspection](https://www.linkedin.com/pulse/computer-vision-quality-inspection-how-ai-improves-accuracy-compliance-2v3kf)

**Engineering insight:** Higher mAP is valuable, but evaluate the model using the metric and IoU threshold relevant to your actual application.

---

## 2. Latency

<badge color="success">Very Important</badge>

Latency is the time taken by a model to process an input and produce its prediction.

For object detection, latency is generally measured in milliseconds per image or frame.

$$
Latency = T_{\text{output}}-T_{\text{input}}
$$

For example:

| Model | Inference latency |
|---|---:|
| Model A | 10 ms |
| Model B | 25 ms |
| Model C | 80 ms |

Lower latency generally means faster predictions.

### 2.1 What is FPS?

FPS means **Frames Per Second**.

It measures how many frames a system processes per second.

For a simple sequential inference pipeline:

$$
FPS \approx \frac{1000}{Latency(ms)}
$$

Example:

If a model takes 20 ms to process one frame:

$$
FPS \approx \frac{1000}{20}=50
$$

**50 FPS**

![Real-time object detection on NVIDIA Jetson Orin NX](https://images.openai.com/static-rsc-4/Dx863E-AAzVKb_hkdtJxZ4I2BzBB2Fdddczyo0ft-TEbZDQdPDY-gSq114mqdljdEDJljmhTCvywhFFVhJGBXrwMIwNaCnZyuA627zrh4hgxTk62XqUc_b_NTo9FFQUopnUGqP6xJVcobpM5TXaK5Iwc_vwkhMRXCkKDCbEMkRk?purpose=inline)

[Source: NVIDIA Jetson Orin NX Real-time Object Detection Demonstration](https://manuals.plus/video/24ab2fbb04fb64e8ed1a91dd4464b8948f5bbeec1dd69b4e0ac0aa0b196d3133)

### 2.2 Why does latency matter?

Imagine an autonomous vehicle travelling at 60 km/h.

Convert speed into metres per second:

$$
60 \times \frac{1000}{3600}=16.67\ m/s
$$

During a 100 ms detection delay, the vehicle travels approximately:

$$
16.67 \times 0.1=1.667\ m
$$

That's why latency is particularly important in real-time systems.

**Important:** Model inference time is not the entire system response time. Camera capture, image preprocessing, data transfer and post-processing also contribute to end-to-end latency.

---

## 3. Model Size

Model size refers to how much storage space the trained model requires.

It is influenced by:

- Number of parameters.
- Numerical precision (FP32, FP16, INT8, etc.).
- Model architecture.
- Stored model weights.

### 3.1 Understanding parameters

A neural network contains learnable parameters called weights and biases.

Generally, more parameters mean a larger model file, although the exact size depends on parameter precision and storage format.

For example, a model with 25 million parameters stored in FP32 requires approximately:

$$
25,000,000 \times 4 = 100,000,000\ bytes
$$

That's approximately 100 MB in decimal units for the weights alone.

If stored in FP16, each parameter uses 2 bytes:

$$
25,000,000 \times 2 = 50,000,000\ bytes
$$

![Neural network model size comparison](https://images.openai.com/static-rsc-4/PLI8-ghd3ObSzwwXTwSrdOqcudW0FuALqRqdc-1VICEQ0GA20Jm6VD2eX2SI0e2KD_KbUgMWUGH9u4vqmkltTt4lLJ5VRQTTPvQ2OE4km8rXJ8E6vRwrvGr4MeleNKhcwb9zhcRsl7yBjQ22rfskdn9gJzuXpnd89NtOOGTg_zU?purpose=inline)

[Source: ImageNet Network Benchmark Comparison](https://www.cnblogs.com/shine-lee/p/11426134.html)

### 3.2 Why does model size matter?

Imagine deploying an object detector on a small embedded device.

A large model may create problems:

- Limited storage.
- High memory consumption.
- Longer model loading time.
- Greater computational requirements.
- Difficult deployment on edge devices.

A smaller model may be preferable even if it sacrifices some accuracy.

### 3.3 Model size vs inference memory

One important distinction:

**Model file size is not the same as inference memory usage.**

During inference, the system also needs memory for:

- Intermediate feature maps.
- Activations.
- Input images.
- Temporary tensors.
- Runtime operations.

Therefore, a model with a small file size can still require substantial RAM or GPU memory.

Remember: Model size tells you how much space the weights occupy. Runtime memory tells you how much memory the complete inference process needs.

---

## 4. Hardware

The best architecture also depends on where you plan to run it.

Consider four deployment environments.

### A. High-performance GPU

![High-performance multi-GPU workstation](https://images.openai.com/static-rsc-4/MaxxSCUHT24ix8Ovg8fVOAb_XS3FX_9z2ZJd4ZD7Eqgf90e1iJwypJ0jPmAyPS2RXYKsd5UoFsxxrsFTG5zuix_TIT8_5rT3_uDR_4M43HZphLfIIFpGmznGWKqlDf211c90vmzPnmC4y8A9-WnAw_eHMAr9o86txyPd41keMTQ?purpose=inline)

[Source: AIME G400 Workstation](https://www.aime.info/shop/product/aime-g400-workstation)

Examples:

- NVIDIA RTX GPUs.
- NVIDIA data-centre GPUs.

Suitable for:

- Large detection models.
- High-resolution images.
- Complex architectures.
- Model training and fine-tuning.

### B. CPU-only system

![Industrial CPU-based machine vision system](https://images.openai.com/static-rsc-4/2gOaCRfM8Ec32bsim0UhLK_DNJojhStoXUZht_ZdmnUM7QrgDRsZMTzXWaqBk6n6d25WsjV1uWE9vfm-pU4Suk1nH2QX_XioHwV3OU2fapyuDy5dIVnyJ3YxLE9ue1qgo_30dvCEm4r_caDN7KHY6BeFgZJRa2XudS90OCsse1o?purpose=inline)

[Source: Cincoze — Industrial AOI Inspection System](https://www.cincoze.com/zh-tw/story_info.php?id=106)

Suitable when:

- No dedicated GPU is available.
- Detection speed requirements are moderate.
- The model is relatively lightweight.
- Hardware cost and power consumption are important.

Small, optimized models are often preferable here.

### C. Edge devices

![NVIDIA Jetson Orin Nano edge AI device](https://images.openai.com/static-rsc-4/EbCNtRbyGcszU-nygcR1VBIw_nFOScw31sVGg5bH2V64SsyNMiOiKirWfBSGR45ESRlhTv4VTnqFiNZAOWcvvPUCuE5sShn58X9RK_TRkk0sBeZk61Al8ydbpBEmtqGWuuntOuqJMbcDZrOj-GRb8Ucc4gMBNqLHEVUKsdGVgTU?purpose=inline)

[Source: Seeed Studio — reComputer J3011](https://www.tinytronics.nl/en/development-boards/single-board-computers/seeed-studio-recomputer-j3011-edge-ai-device-with-nvidia-jetson-orin-nano-8gb-module)

Examples:

- NVIDIA Jetson devices.
- Raspberry Pi with suitable accelerators.
- Industrial edge computers.

Used in:

- Robotics.
- Smart cameras.
- Factory automation.
- Autonomous systems.

### D. Cloud deployment

![Cloud AI infrastructure](https://images.openai.com/static-rsc-4/gU_0MyVThaMgxvZnpbtFzkwPDW63u8K_VVPu38kyNKfGtm-O8yLuyovC5R8zyH0fgPkAp_cRm5l_sossJ21EyjixATCg_k2E3n8QGh-D91fdbJeda_S7fWsJR0411rMuYd1OwA0DJIqLY-8bUFLoLJZQJ-aIMXOJMBKiRUvUDh8?purpose=inline)

[Source: AI Cloud Infrastructure](https://www.patriotictech.com/ai-cloud-infrastructure-why-companies-are-renting-gpus-and-tpus-instead-of-buying-them/)

Suitable for:

- Large-scale image processing.
- Centralized inference services.
- Applications requiring scalable computing resources.
- Multiple users and cameras.

### Hardware selection table

| Hardware | Main consideration |
|---|---|
| High-performance GPU | High throughput and complex models |
| CPU | Lightweight models and optimized inference |
| Edge device | Power, memory and latency constraints |
| Cloud GPU | Scalability, throughput and operating cost |

**Engineering insight:** Don't choose the architecture first and then hope the hardware can handle it. Identify the deployment hardware and benchmark compatible models.

---

## 5. Dataset Size

The dataset is another important factor in model selection.

Imagine two projects.

**Project A — Vehicle detection**

- 50,000 labelled images.
- Different weather conditions.
- Day and night images.
- Different vehicle types.
- Multiple camera angles.

**Project B — Industrial defect detection**

- 500 labelled images.
- Only a few examples of defective products.
- One manufacturing environment.
- Limited defect variations.

Would you approach both projects in exactly the same way?

Not necessarily.

### 5.1 Small Dataset

When labelled data is limited:

- Prefer a model with useful pretrained weights.
- Use transfer learning.
- Apply suitable data augmentation.
- Carefully split training, validation and test data.
- Monitor overfitting.

![Random rotation data augmentation](https://images.openai.com/static-rsc-4/ZYW4CuwLP8gF7RSI0EQgmCjsUDwsLpZ7DerAJ2E7gY45BfdAd5Efu0a5Pv81ea9UIK1WKwv1JyDk-5OXVGb19c9ZGf7v51jhDDcl-zWamvi-LmH2fri9z21y0b867Pccgc2UKZLK3iqOzVtWYRW9fkwkwxLbPptDt9cYvzxGOms?purpose=inline)

[Source: Roboflow — Random Rotate Data Augmentation](https://blog.roboflow.com/why-and-how-to-implement-random-rotate-data-augmentation/)

For example, instead of training a YOLO model completely from scratch using 500 images, you could fine-tune a pretrained model on your custom dataset.

This allows the model to reuse visual features learned from earlier training.

### 5.2 Large Dataset

When a large, diverse labelled dataset is available:

- You can explore larger model architectures.
- You can train or fine-tune more extensively.
- You can compare different detection architectures.
- You can investigate performance across different object categories.

However, more images do not automatically guarantee better performance.

**Dataset quality matters just as much as dataset quantity.**

Important factors include:

- Correct bounding-box annotations.
- Class balance.
- Object-size distribution.
- Lighting conditions.
- Background variation.
- Camera angles.
- Training and deployment data similarity.

### 5.3 Dataset size and model selection

| Dataset situation | Practical approach |
|---|---|
| Very small dataset | Pretrained lightweight model + transfer learning |
| Medium dataset | Fine-tune pretrained models and compare architectures |
| Large diverse dataset | Evaluate larger models and broader architecture choices |
| Imbalanced dataset | Consider class-aware sampling, augmentation and suitable loss functions |

Remember that these are starting strategies, not strict rules. A small but highly diverse dataset can sometimes be more useful than a much larger dataset with poor annotations.

---

## 6. Real-Time Requirement

<badge color="success">Very Important</badge>

This factor connects all the previous concepts.

A real-time system must process incoming data quickly enough to meet its application deadline.

Consider three applications.

### Scenario A — Real-time traffic monitoring

![Smart city traffic monitoring](https://images.openai.com/static-rsc-4/AoDhvPVvky0G9uuFNuKgT_I9IWoNBXE-4yQt821MS-i-MvBR7QStHpe0fSV3KxrP2ZQ1a1sLB2nibT3Af0fgX12-8USU5Sdk2eBywWl_9xD7TPWDRIQI4Mc1q9VsK5-RzsbPe4rWULaMzcrt5P-mR--0_CrP2tegFOidOuCbs_M?purpose=inline)

[Source: Katomaran — Smart City Video Intelligence](https://www.katomaran.com/industries/smart-cities-solutions)

Requirements:

- Detect multiple vehicles.
- Process continuous video.
- Produce predictions with low latency.
- Operate reliably for long periods.

A lightweight, optimized YOLO-style detector could be a sensible starting point.

### Scenario B — Industrial defect inspection

![Industrial PCB quality inspection](https://images.openai.com/static-rsc-4/fprLk6b5cjj_di1Lh070w-HKq_9AUAkrTmbyjfYxxmb6v8D5taH3U1O0VBITbeiv03MSYIRtR6CFOincIbBRafhqPpIEQ_XA9vawOA-6fZcYegpQRcTip4-dr337phKM7hiLf48ZNDVpLUtw5ILSU1Dl9stQoCP0zILsUhfWtdo?purpose=inline)

[Source: Optimizing Quality Through Precision PCBA Services](https://www.josephsan.com.sg/post/optimizing-quality-through-precision-pcba-services)

Requirements:

- Detect tiny defects.
- Maintain high localization quality.
- Process products at a defined production-line speed.
- Minimize missed defects.

Here, accuracy and small-object detection may be more important than maximizing FPS.

A carefully configured Faster R-CNN, RetinaNet or suitable YOLO model could be evaluated.

### Scenario C — Offline medical image analysis

![Medical lesion detection](https://images.openai.com/static-rsc-4/qtuBixOYETZ_zn7t-g81UVyEyh2Dbpnx5oKx6cg-JfznekfIzu3ISrCWMENajmwsioeGPdwt0Vkj1oayyOLMWpzM7ZN1IkWPS9Ry7dPjPDBAHFcHpLndMK35KqIJIO6xLxET78Pl5k4Nb4Q0Lf6PQiaS8WgJFub1zPxr57wC8LE?purpose=inline)

[Source: DeepLesion — Universal Lesion Detection](https://www.spiedigitallibrary.org/journals/journal-of-medical-imaging/volume-5/issue-03/036501/DeepLesion--automated-mining-of-large-scale-lesion-annotations-and/10.1117/1.JMI.5.3.036501.full)

Requirements:

- Accurate localization.
- Detailed image analysis.
- Processing time may be less restrictive.
- Reliability and validation are critical.

A slower architecture may be acceptable if it offers meaningful performance improvements for the task.

These are candidate starting points, not universal recommendations. Medical AI also requires domain-specific validation and regulatory considerations.

---

## 7. Practical Model Selection — Decision Framework

Now let's combine all six factors into one engineering decision.

| Requirement | Possible starting choice | Why? |
|---|---|---|
| Real-time detection | Lightweight YOLO | Designed for efficient detection |
| Detailed region detection | Faster R-CNN | Two-stage proposal and refinement |
| Multi-scale detection | SSD / RetinaNet | Predictions across feature scales |
| Foreground-background imbalance | RetinaNet | Focal Loss |
| Transformer-based detection | DETR / variants | Set prediction using attention |
| Low-power edge deployment | Compact, optimized detector | Lower compute and memory requirements |
| Small custom dataset | Pretrained detector + fine-tuning | Reuses learned visual features |

### Model selection flowchart

![Computer vision deployment benchmarking](https://images.openai.com/static-rsc-4/ncbrxUw9T4boFCP6NQrgQIbrB-UbGJMyfJQTu7t-2OAAsjBvMrvdUE_EC4nI7a3OOHby1x0elaUQSlSsptEtPXsDOM0GlENAEgw0xuvyY60mi-YLDxakbD2zJonsIvJ-LmOAWZ0pPd6HmlHs_hfLkMD0ct7guTXLcDQHxYhoz5Q?purpose=inline)

[Source: RoboLabs — Benchmarking Reality Check](https://www.robolabs.ai/fr/resources/blog/benchmarking-reality-check-cv-model-deployment)

```text
              Start
                |
                ▼
        What is the application?
                |
         ┌──────┴───────┐
         ▼              ▼
     Real-time      Accuracy-focused
         |              |
         ▼              ▼
   Lightweight      Compare localization
    detectors       and detection quality
         |              |
         └──────┬───────┘
                ▼
       Identify deployment
             hardware
                |
                ▼
        Check dataset size
        and annotation quality
                |
                ▼
       Train / Fine-tune Models
                |
                ▼
       Evaluate mAP and Recall
                |
                ▼
       Benchmark Latency,
       Memory and Model Size
                |
                ▼
         Select Best Model
```

---

## 8. Industry Example — Industrial Defect Detection

Let's apply everything to an engineering problem.

Imagine you're developing a vision system for an electronics manufacturing company.

The system must identify:

- Missing components.
- Incorrect component placement.
- Damaged PCB tracks.
- Soldering defects.

![PCB component presence detection](https://images.openai.com/static-rsc-4/_uc5I9o_4l14WSImqoqzlbj13EkJI6TtrAsLDjhVzec6qMZHpurY5Q4Z0xIVaJWmjCxPcxZfDiSjOrEj0DkyVpfFdQNPdKdZn6919470w0WAqeonbTkccF3rDnf2VA1fbnnQ3upcMmn2LZiiQPUw9yHckbFp80d-AqN207l843k?purpose=inline)

[Source: Component Presence Detection in Electronics Assembly Lines](https://intelgic.com/component-presence-detection-electronics-assembly)

Suppose the project requirements are:

| Requirement | Project specification |
|---|---|
| Application | PCB defect detection |
| Dataset | 2,000 labelled images |
| Deployment | Industrial edge computer |
| Processing | Production-line inspection |
| Main priority | Detect small defects |
| Latency | Must meet the conveyor's inspection deadline |

### How would we select a model?

**Step 1 — Dataset**

2,000 images is a relatively small dataset.

Start with pretrained models and fine-tune them using the labelled PCB images.

**Step 2 — Accuracy**

Prioritize recall for important defect classes so that fewer defective products are missed. Also monitor precision to avoid excessive false alarms.

**Step 3 — Model architecture**

Evaluate a compact YOLO model and a suitable two-stage detector such as Faster R-CNN.

Don't assume one will always win.

**Step 4 — Hardware**

Benchmark both on the actual industrial computer.

**Step 5 — Final selection**

Choose the model that meets the defect-detection quality requirements while satisfying the production-line latency and memory constraints.

This is how an ML engineer makes a practical architecture decision.

---

## 9. Model Benchmarking — What Should You Measure?

One important professional habit: **never compare two models using only their reported mAP values.**

Use the same:

- Test dataset.
- Input image resolution.
- Evaluation protocol.
- Hardware.
- Batch size.
- Precision mode.
- Preprocessing and post-processing conditions.

Then compare:

| Metric | What to measure |
|---|---|
| mAP@0.50 | Detection performance at IoU 0.50 |
| mAP@0.50:0.95 | Detection performance across multiple IoU thresholds |
| Precision | False-positive behaviour |
| Recall | Missed-object behaviour |
| Latency | Processing time |
| FPS | Throughput |
| Model size | Weight storage requirements |
| Peak memory | Runtime memory requirement |

A very important distinction: FPS is throughput, while latency is the time taken for an individual request. In a pipelined or batched system, they are not necessarily exact reciprocals.

### An example comparison

The following is an illustrative benchmark, not measured model performance.

| Model | mAP | Latency | Model size |
|---|---:|---:|---:|
| A | 92% | 80 ms | 150 MB |
| B | 88% | 20 ms | 35 MB |
| C | 84% | 8 ms | 12 MB |

Illustrative mAP values for three hypothetical detectors, not real benchmark results.

### Which model should we choose?

- If accuracy is the main priority and latency is acceptable: Model A.
- If we need a balance between accuracy and speed: Model B.
- If we have strict edge-device constraints: Model C.

There is no universally best model. The best model is the one that satisfies the application's requirements.

---


## What You Should Remember

The six factors for practical model selection:

1. **Accuracy:** How well does the model detect and localize objects?
2. **Latency:** How quickly does it produce predictions?
3. **Model size:** How much storage do the model weights require?
4. **Hardware:** Where will the model run?
5. **Dataset size:** How much labelled training data is available, and how good is it?
6. **Real-time requirement:** Does the application need immediate predictions or can it process images offline?

The engineering principle:

> Don't choose the most powerful model. Choose the model that delivers the required detection quality within your hardware, latency, memory and deployment constraints.

---